# 04-4. 본문·요약 완전 중복 제거

04-3 최종 clean 데이터에서 모델 입력·정답인 `document_original + summary`가 완전히 같은 행을 전역으로 찾는다. 제목·링크와 무관하게 하나만 남긴다. split을 넘는 그룹은 test → validation → train 우선순위로 대표 행을 보존해 원본 test를 최대한 유지한다.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT = Path(r"C:\Users\KDS-30\Desktop\Projects\06_Team_Project4")
assert Path(sys.prefix).resolve() == (PROJECT / ".venv").resolve(), "커널을 Python 3.11 (Team4 News GPU)로 바꾸세요."
INPUT_DIR = PROJECT / "processed_data" / "repeat_4gram_excluded_document_cleaned"
OUTPUT_DIR = PROJECT / "processed_data" / "final_clean_exact_deduplicated"
REVIEW_DIR = PROJECT / "review_data" / "04-4_exact_duplicate_dedup"
assert INPUT_DIR.exists(), "먼저 04-3을 실행하세요."
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
print("입력:", INPUT_DIR)
print("중복 제거본:", OUTPUT_DIR)
print("검토 결과:", REVIEW_DIR)

입력: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\processed_data\repeat_4gram_excluded_document_cleaned
중복 제거본: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\processed_data\final_clean_exact_deduplicated
검토 결과: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\review_data\04-4_exact_duplicate_dedup


In [2]:
split_order = ["train", "validation", "test"]
frames = [pd.read_csv(INPUT_DIR / f"{split}.csv") for split in split_order]
all_df = pd.concat(frames, ignore_index=True)
assert all_df["row_id"].is_unique

# document는 URL/저작권 문구가 제거된 버전이므로, 중복 판정에는 정제 전 원문을 사용합니다.
KEY_COLUMNS = ["document_original", "summary"]
assert all_df[KEY_COLUMNS].notna().all().all()

group_size = all_df.groupby(KEY_COLUMNS, dropna=False)["row_id"].transform("size")
all_df["exact_duplicate_group_size"] = group_size
all_df["is_exact_duplicate"] = group_size.gt(1)

# test를 우선 보존하고, 같은 split 안에서는 원래 index가 가장 작은 행을 대표로 남깁니다.
split_priority = {"test": 0, "validation": 1, "train": 2}
all_df["_split_priority"] = all_df["split"].map(split_priority)
all_df["_original_position"] = np.arange(len(all_df))
ordered = all_df.sort_values(["_split_priority", "source_index", "_original_position"]).copy()
ordered["keep_exact_duplicate"] = ~ordered.duplicated(KEY_COLUMNS, keep="first")
all_df = all_df.merge(ordered[["row_id", "keep_exact_duplicate"]], on="row_id", how="left", validate="one_to_one")
all_df["duplicate_action"] = np.where(all_df["keep_exact_duplicate"], "keep", "exclude_exact_duplicate")

summary = (all_df.groupby("split", sort=False).agg(
    input_rows=("row_id", "size"),
    duplicate_rows=("is_exact_duplicate", "sum"),
    excluded_rows=("keep_exact_duplicate", lambda x: int((~x).sum())),
    output_rows=("keep_exact_duplicate", "sum"),
)
    .reset_index())

cross_split_groups = (all_df.loc[all_df["is_exact_duplicate"]]
    .groupby(KEY_COLUMNS, dropna=False)["split"].nunique().gt(1).sum())
display(summary)
print("split을 넘는 완전 중복 그룹:", f"{cross_split_groups:,}")

,split,input_rows,duplicate_rows,excluded_rows,output_rows
0,train,21073,2436,1831,19242
1,validation,2345,278,106,2239
2,test,2606,285,42,2564


split을 넘는 완전 중복 그룹: 404


In [3]:
# 대표 행만 새 clean split에 저장합니다. 원본·제외 행은 삭제하지 않고 검토 CSV에 남깁니다.
for split, frame in all_df.groupby("split", sort=False):
    output = frame.loc[frame["keep_exact_duplicate"]].copy()
    output = output.drop(columns=["_split_priority", "_original_position"])
    output.to_csv(OUTPUT_DIR / f"{split}.csv", index=False, encoding="utf-8-sig")

excluded_columns = [
    "row_id", "split", "source_index", "link", "date", "category", "press",
    "title", "document_original", "summary", "exact_duplicate_group_size",
    "duplicate_action",
]
excluded = all_df.loc[~all_df["keep_exact_duplicate"], excluded_columns].copy()
excluded["review_decision"] = "exclude_exact_duplicate"
excluded["memo"] = ""
excluded.to_csv(REVIEW_DIR / "excluded_exact_duplicate_rows.csv", index=False, encoding="utf-8-sig")
summary.to_csv(REVIEW_DIR / "split_summary.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([{
    "input_data": str(INPUT_DIR),
    "output_data": str(OUTPUT_DIR),
    "duplicate_key": "document_original + summary (exact match; title/link ignored)",
    "representative_priority": "test, validation, train; then lowest source_index",
    "cross_split_duplicate_groups": int(cross_split_groups),
}]).to_csv(REVIEW_DIR / "run_info.csv", index=False, encoding="utf-8-sig")

# 저장된 데이터에는 같은 키가 두 번 남지 않아야 합니다.
for split in split_order:
    check = pd.read_csv(OUTPUT_DIR / f"{split}.csv")
    assert not check.duplicated(KEY_COLUMNS).any(), f"{split} 안에 완전 중복이 남았습니다."

print("저장 완료")
print("-", REVIEW_DIR / "excluded_exact_duplicate_rows.csv")
print("-", OUTPUT_DIR / "train.csv")

저장 완료
- C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\review_data\04-4_exact_duplicate_dedup\excluded_exact_duplicate_rows.csv
- C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\processed_data\final_clean_exact_deduplicated\train.csv


학습에는 `final_clean_exact_deduplicated`의 train·validation·test를 사용한다. `excluded_exact_duplicate_rows.csv`는 링크만 달랐던 제외 행의 감사 목록이다.